In [16]:
import os

# SSL certificate configuration:
# Uses the Linux system's trusted CA certificates for HTTPS connections.
# This is required in our corporate network because the default Python
# certificate bundle does not trust the network's certificate chain.
os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/certs/ca-certificates.crt'
os.environ['SSL_CERT_FILE'] = '/etc/ssl/certs/ca-certificates.crt'

# Mini RAG Chatbot

## 1. Project Overview

This project implements a simple Retrieval-Augmented Generation (RAG) chatbot that answers questions using information retrieved from a PDF document.

The pipeline follows:

PDF → Text Extraction → Chunking → Embeddings → Vector Search → Context Retrieval → LLM → Answer

In [3]:
# Import only required class from the module to open and read pdf files
from pypdf import PdfReader

In [4]:
# Pdf path
pdf_path = '/home/nineleaps/Documents/da_python/.venv/GenAI/mini-rag-chatbot/sample_data/sample.pdf'

In [5]:
# Read the pdf
reader = PdfReader(pdf_path)

In [6]:
# Verify
print(f'Number of pages: {len(reader.pages)}')

Number of pages: 35


In [7]:
# Extract content from the pdf
text = ''

for page in reader.pages:
    text += page.extract_text() + '\n'

In [8]:
# Inspect
print(text[:2000])

 

PENGUIN   READERS  2000  
 
 
  
 
  
 
www.penguinreaders.com
 
 
The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter 
Pearson Education Limited                                                                            
Edinburgh Gate, Harlow,                                                                               
Essex CM20 2JE, England                                                                              
and Associated Companies throughout the world. 
ISBN 0 582 41923 9 
 
First published 1876                                                                                  
Published by Puffin Books 1950                                                                         
This edition first published 2000 
Copyright © Penguin Books 2000 
Typeset by Digital Type, London 
Set in 12/14ptBembo 
Printed in Spain by Mateu 

## 5. Split Text into Chunks

Large documents are divided into smaller pieces called chunks so that relevant sections can be retrieved efficiently during semantic search.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter:
# A LangChain text-splitting tool that divides large text into smaller,
# meaningful pieces while trying to keep related text together.
#
# chunk_size:
# The maximum target size of each chunk, measured in characters.
#
# chunk_overlap:
# The number of characters shared between consecutive chunks.
# Overlap helps preserve context when important information falls
# near the boundary between two chunks.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

/home/nineleaps/Documents/da_python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Split the extracted PDF text into smaller chunks for later semantic search.
chunks = text_splitter.split_text(text)

In [11]:
# Count how many text chunks were created from the PDF.
print(f'Number of chunks: {len(chunks)}')

Number of chunks: 65


In [12]:
# Display the first chunk to verify that the text was split correctly.
print(chunks[0])

PENGUIN   READERS  2000  
 
 
  
 
  
 
www.penguinreaders.com
 
 
The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter 
Pearson Education Limited                                                                            
Edinburgh Gate, Harlow,


In [13]:
# Display the first three chunks to inspect their size and content.
for i, chunk in enumerate(chunks[:3]):
    print(f'\n--- Chunk {i + 1} ---')
    print(chunk)


--- Chunk 1 ---
PENGUIN   READERS  2000  
 
 
  
 
  
 
www.penguinreaders.com
 
 
The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter 
Pearson Education Limited                                                                            
Edinburgh Gate, Harlow,

--- Chunk 2 ---
Essex CM20 2JE, England                                                                              
and Associated Companies throughout the world. 
ISBN 0 582 41923 9 
 
First published 1876                                                                                  
Published by Puffin Books 1950                                                                         
This edition first published 2000 
Copyright © Penguin Books 2000 
Typeset by Digital Type, London 
Set in 12/14ptBembo

--- Chunk 3 ---
Set in 12/14ptBembo 
Printed in Spain by Mateu Cromo, S. A

In [14]:
# Check the character length of each chunk to understand the chunking result.
chunk_lengths = [len(chunk) for chunk in chunks]

print(f'Minimum chunk length: {min(chunk_lengths)}')
print(f'Maximum chunk length: {max(chunk_lengths)}')
print(f'Average chunk length: {sum(chunk_lengths) / len(chunk_lengths):.2f}')

Minimum chunk length: 271
Maximum chunk length: 498
Average chunk length: 463.86


## 6. Generate Embeddings

Embeddings convert text into numerical vectors that represent semantic meaning.

These vectors allow us to compare the meaning of a user's question with the meaning of the document chunks.

Embeddings give text a location on that conceptual map.

Later, when the user asks a question, we'll create an embedding for the question and look for chunks whose locations are closest.

That's how semantic search will work.

In [17]:
from sentence_transformers import SentenceTransformer

# SentenceTransformer:
# A pretrained model that converts text into numerical embedding vectors.
#
# Embedding:
# A numerical representation of text that captures semantic meaning.
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4375.31it/s]


In [18]:
# Convert one document chunk into a numerical embedding vector.
sample_embedding = embedding_model.encode(chunks[0])

print(f'Embedding dimensions: {len(sample_embedding)}')
print(f'First 10 values: {sample_embedding[:10]}')

Embedding dimensions: 384
First 10 values: [-0.03210997 -0.05208552  0.02694087  0.03203892  0.00049999  0.02150638
 -0.05120209 -0.02016692 -0.02146922  0.0350961 ]


In [19]:
# Generate embeddings for every chunk in the PDF.
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

# Check the number of chunks and the size of each embedding vector.
print(f'Embedding shape: {embeddings.shape}')

Batches: 100%|██████████| 3/3 [00:01<00:00,  1.96it/s]

Embedding shape: (65, 384)
